# Module 2: Data Loading and Augmentation Using Keras
## Image Data Processing and Augmentation Tasks

## Setup and Imports

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import random
from PIL import Image
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# Define paths
base_path = './images_dataSAT/'
class_0_path = os.path.join(base_path, 'class_0_non_agri')
class_1_path = os.path.join(base_path, 'class_1_agri')

# Check if directories exist
print(f"Class 0 (Non-Agriculture) exists: {os.path.exists(class_0_path)}")
print(f"Class 1 (Agriculture) exists: {os.path.exists(class_1_path)}")

# Image extensions to look for
image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.tif', '*.tiff', '*.bmp']

## Task 1: Create 'all_image_paths' list containing paths from both folders

In [ ]:
# Task 1: Create all_image_paths list

all_image_paths = []

# Function to get all image paths from a directory
def get_image_paths(directory):
    paths = []
    for ext in image_extensions:
        paths.extend(glob.glob(os.path.join(directory, ext)))
        paths.extend(glob.glob(os.path.join(directory, ext.upper())))
    return paths

# Get paths from both directories
class_0_paths = get_image_paths(class_0_path)
class_1_paths = get_image_paths(class_1_path)

# Combine all paths
all_image_paths = class_0_paths + class_1_paths

# Sort for consistency (optional)
all_image_paths.sort()

# Display results
print(f"=" * 60)
print(f"TASK 1: Created all_image_paths list")
print(f"=" * 60)
print(f"Total images found: {len(all_image_paths)}")
print(f"Class 0 (Non-Agriculture) images: {len(class_0_paths)}")
print(f"Class 1 (Agriculture) images: {len(class_1_paths)}")

# Show sample paths
print("\nSample paths (first 3 from each class):")
print("\nClass 0 samples:")
for i, path in enumerate(class_0_paths[:3]):
    print(f"  {i+1}. {os.path.basename(path)}")

print("\nClass 1 samples:")
for i, path in enumerate(class_1_paths[:3]):
    print(f"  {i+1}. {os.path.basename(path)}")

## Task 2: Create temporary list with image paths and labels, then randomly select 5

In [ ]:
# Task 2: Create temp list with paths and labels, then select 5 random samples

# Create labels (0 for non-agriculture, 1 for agriculture)
labels = [0] * len(class_0_paths) + [1] * len(class_1_paths)

# Create temp list by binding paths and labels using zip
temp = list(zip(all_image_paths, labels))

# Randomly select 5 samples
random.seed(42)  # For reproducibility
random_samples = random.sample(temp, min(5, len(temp)))

# Display results
print(f"=" * 60)
print(f"TASK 2: Randomly selected 5 image paths with labels")
print(f=" * 60)

print(f"\nRandomly selected samples:")
for i, (path, label) in enumerate(random_samples, 1):
    class_name = "Agriculture" if label == 1 else "Non-Agriculture"
    print(f"\nSample {i}:")
    print(f"  Path: {path}")
    print(f"  Filename: {os.path.basename(path)}")
    print(f"  Label: {label} ({class_name})")

# Verify distribution
print("\n" + "=" * 60)
print("Distribution in temp list:")
print(f"Total samples in temp: {len(temp)}")
print(f"Class 0 samples: {sum(1 for _, label in temp if label == 0)}")
print(f"Class 1 samples: {sum(1 for _, label in temp if label == 1)}")

# Display the random samples as images
print("\nDisplaying randomly selected images:")
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, (path, label) in enumerate(random_samples):
    img = Image.open(path)
    axes[i].imshow(img)
    class_name = "Agriculture" if label == 1 else "Non-Agriculture"
    axes[i].set_title(f'{class_name}\n{os.path.basename(path)[:20]}...')
    axes[i].axis('off')
plt.suptitle('Randomly Selected Sample Images', fontsize=16)
plt.tight_layout()
plt.show()

## Custom Data Generator Function

In [ ]:
# Define custom data generator function

def custom_data_generator(image_paths, labels, batch_size=8, target_size=(224, 224), 
                          augment=False, shuffle=True):
    """
    Custom data generator for image loading and augmentation
    
    Args:
        image_paths: List of image file paths
        labels: List of corresponding labels
        batch_size: Number of images per batch
        target_size: Tuple of (height, width) to resize images
        augment: Boolean to apply augmentation
        shuffle: Boolean to shuffle data
    
    Yields:
        Tuple of (batch_images, batch_labels)
    """
    # Create augmentation generator if needed
    if augment:
        datagen = ImageDataGenerator(
            rotation_range=20,
            width_shift_range=0.2,
            height_shift_range=0.2,
            shear_range=0.2,
            zoom_range=0.2,
            horizontal_flip=True,
            fill_mode='nearest'
        )
    
    num_samples = len(image_paths)
    indices = np.arange(num_samples)
    
    while True:
        if shuffle:
            np.random.shuffle(indices)
        
        for start_idx in range(0, num_samples, batch_size):
            end_idx = min(start_idx + batch_size, num_samples)
            batch_indices = indices[start_idx:end_idx]
            
            batch_images = []
            batch_labels = []
            
            for idx in batch_indices:
                # Load and preprocess image
                img_path = image_paths[idx]
                try:
                    img = Image.open(img_path)
                    
                    # Convert to RGB if necessary
                    if img.mode != 'RGB':
                        img = img.convert('RGB')
                    
                    # Resize image
                    img = img.resize(target_size)
                    
                    # Convert to numpy array and normalize
                    img_array = np.array(img) / 255.0
                    
                    batch_images.append(img_array)
                    batch_labels.append(labels[idx])
                    
                except Exception as e:
                    print(f"Error loading image {img_path}: {e}")
                    continue
            
            # Convert to numpy arrays
            batch_images = np.array(batch_images)
            batch_labels = np.array(batch_labels)
            
            # Apply augmentation if requested
            if augment and len(batch_images) > 0:
                augmented_images = []
                augmented_labels = []
                
                for img, label in zip(batch_images, batch_labels):
                    # Add batch dimension for augmentation
                    img_batch = np.expand_dims(img, 0)
                    
                    # Apply augmentation
                    aug_iter = datagen.flow(img_batch, batch_size=1)
                    aug_img = next(aug_iter)[0]
                    
                    augmented_images.append(aug_img)
                    augmented_labels.append(label)
                
                batch_images = np.array(augmented_images)
                batch_labels = np.array(augmented_labels)
            
            yield batch_images, batch_labels

## Task 3: Generate a batch of data (batch size = 8) using custom_data_generator

In [ ]:
# Task 3: Generate one batch of data

print(f"=" * 60)
print(f"TASK 3: Generate a batch of data (batch size = 8)")
print(f"=" * 60)

# Create generator without augmentation for task 3
batch_size = 8
data_gen = custom_data_generator(
    image_paths=all_image_paths,
    labels=labels,
    batch_size=batch_size,
    target_size=(224, 224),
    augment=False,
    shuffle=True
)

# Generate one batch
batch_images, batch_labels = next(data_gen)

# Display batch information
print(f"\nBatch generated successfully!")
print(f"Batch images shape: {batch_images.shape}")
print(f"Batch labels shape: {batch_labels.shape}")
print(f"Image data type: {batch_images.dtype}")
print(f"Image value range: [{batch_images.min():.3f}, {batch_images.max():.3f}]")

# Show label distribution in batch
unique, counts = np.unique(batch_labels, return_counts=True)
label_dist = dict(zip(unique, counts))
print(f"\nLabel distribution in batch:")
for label, count in label_dist.items():
    class_name = "Agriculture" if label == 1 else "Non-Agriculture"
    print(f"  Class {label} ({class_name}): {count} images")

# Display the batch images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(batch_size):
    axes[i].imshow(batch_images[i])
    class_name = "Agriculture" if batch_labels[i] == 1 else "Non-Agriculture"
    axes[i].set_title(f'Image {i+1}\nLabel: {int(batch_labels[i])} ({class_name})')
    axes[i].axis('off')

plt.suptitle(f'Batch of {batch_size} Images (No Augmentation)', fontsize=16)
plt.tight_layout()
plt.show()

# Show detailed image statistics
print("\n" + "-" * 40)
print("Image Statistics:")
for i in range(min(3, batch_size)):  # Show first 3 images details
    print(f"\nImage {i+1}:")
    print(f"  Shape: {batch_images[i].shape}")
    print(f"  Mean pixel value: {batch_images[i].mean():.3f}")
    print(f"  Std pixel value: {batch_images[i].std():.3f}")

## Task 4: Create validation data using a batch size of 8

In [ ]:
# Task 4: Create validation data

print(f"=" * 60)
print(f"TASK 4: Create validation data (batch size = 8)")
print(f=" * 60)

# Split data into training and validation
validation_split = 0.2
num_samples = len(all_image_paths)
num_val = int(num_samples * validation_split)

# Create indices for validation set
indices = np.arange(num_samples)
np.random.seed(42)
np.random.shuffle(indices)

val_indices = indices[:num_val]
train_indices = indices[num_val:]

# Create validation paths and labels
val_paths = [all_image_paths[i] for i in val_indices]
val_labels = [labels[i] for i in val_indices]

print(f"\nData Split Information:")
print(f"Total samples: {num_samples}")
print(f"Training samples: {len(train_indices)}")
print(f"Validation samples: {len(val_indices)}")

# Create validation data generator
val_batch_size = 8
val_generator = custom_data_generator(
    image_paths=val_paths,
    labels=val_labels,
    batch_size=val_batch_size,
    target_size=(224, 224),
    augment=False,  # No augmentation for validation
    shuffle=False    # No shuffling for validation
)

# Generate validation batches
print(f"\nGenerating validation batches...")
val_batches = []
num_val_batches = (len(val_paths) + val_batch_size - 1) // val_batch_size

for batch_idx in range(min(3, num_val_batches)):  # Show first 3 batches max
    val_images, val_batch_labels = next(val_generator)
    val_batches.append((val_images, val_batch_labels))
    
    print(f"\nValidation Batch {batch_idx + 1}:")
    print(f"  Images shape: {val_images.shape}")
    print(f"  Labels shape: {val_batch_labels.shape}")
    
    # Show label distribution in this validation batch
    unique, counts = np.unique(val_batch_labels, return_counts=True)
    for label, count in zip(unique, counts):
        class_name = "Agriculture" if label == 1 else "Non-Agriculture"
        print(f"    Class {int(label)} ({class_name}): {count} images")

# Display first validation batch
if val_batches:
    val_images, val_batch_labels = val_batches[0]
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for i in range(min(len(val_images), 8)):
        axes[i].imshow(val_images[i])
        class_name = "Agriculture" if val_batch_labels[i] == 1 else "Non-Agriculture"
        axes[i].set_title(f'Val Image {i+1}\nLabel: {int(val_batch_labels[i])} ({class_name})')
        axes[i].axis('off')
    
    # Hide empty subplots if batch size < 8
    for i in range(len(val_images), 8):
        axes[i].axis('off')
    
    plt.suptitle(f'First Validation Batch (Batch Size: {val_batch_size})', fontsize=16)
    plt.tight_layout()
    plt.show()

# Create full validation dataset info
print("\n" + "-" * 40)
print("Validation Dataset Summary:")
print(f"Total validation samples: {len(val_paths)}")
print(f"Total validation batches: {num_val_batches}")
print(f"Class distribution in validation set:")
val_class_0 = sum(1 for label in val_labels if label == 0)
val_class_1 = sum(1 for label in val_labels if label == 1)
print(f"  Class 0 (Non-Agriculture): {val_class_0} images ({val_class_0/len(val_labels)*100:.1f}%)")
print(f"  Class 1 (Agriculture): {val_class_1} images ({val_class_1/len(val_labels)*100:.1f}%)")

## Bonus: Data Augmentation Visualization

In [ ]:
# Bonus: Show augmented data example

print(f"=" * 60)
print("BONUS: Data Augmentation Example")
print(f"=" * 60)

# Create generator with augmentation
aug_gen = custom_data_generator(
    image_paths=all_image_paths[:1] * 8,  # Use same image 8 times for demonstration
    labels=[0] * 8,
    batch_size=8,
    target_size=(224, 224),
    augment=True,
    shuffle=False
)

# Generate augmented batch
aug_images, _ = next(aug_gen)

# Display augmented versions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(8):
    axes[i].imshow(aug_images[i])
    axes[i].set_title(f'Augmented Version {i+1}')
    axes[i].axis('off')

plt.suptitle('Different Augmentations of the Same Image', fontsize=16)
plt.tight_layout()
plt.show()

## Summary of All Tasks

In [ ]:
# Summary of completed tasks
print("=" * 60)
print("SUMMARY OF MODULE 2 TASKS")
print("=" * 60)

# Task 1
print(f"\n✓ Task 1: Created 'all_image_paths' list")
print(f"   - Total paths: {len(all_image_paths)}")
print(f"   - Class 0 paths: {len(class_0_paths)}")
print(f"   - Class 1 paths: {len(class_1_paths)}")

# Task 2
print(f"\n✓ Task 2: Created 'temp' list with zip and selected 5 random samples")
print(f"   - Temp list size: {len(temp)}")
print(f"   - Random samples selected and displayed")

# Task 3
print(f"\n✓ Task 3: Generated batch of data with batch size 8")
print(f"   - Batch images shape: {batch_images.shape}")
print(f"   - Batch labels shape: {batch_labels.shape}")

# Task 4
print(f"\n✓ Task 4: Created validation data with batch size 8")
print(f"   - Validation samples: {len(val_paths)}")
print(f"   - Validation batches: {num_val_batches}")
print(f"   - First validation batch displayed")

print("\n" + "=" * 60)
print("ALL TASKS COMPLETED SUCCESSFULLY!")
print("=" * 60)